In [3]:
#Setup & Data Load


import pandas as pd
import numpy as np
from pathlib import Path

# Define paths
BASE_DIR = Path("/home/niranjanrao07/cod-multiagent-ecommerce/notebooks/data/processed")
INPUT_PATH = BASE_DIR / "amazon_sentiment_subsets" / "amazon_sentiment_cleaned.parquet"
OUTPUT_PATH = BASE_DIR / "amazon_sentiment_subsets" / "amazon_sentiment_mlready.parquet"

# Load cleaned dataset
df = pd.read_parquet(INPUT_PATH)

# Regenerate sentiment label from rating
def map_sentiment(r):
    if r <= 2:
        return "negative"
    elif r == 3:
        return "neutral"
    else:
        return "positive"

df["sentiment"] = df["rating"].apply(map_sentiment)

# Basic info
print("Loaded dataset from:", INPUT_PATH)
print(f"Shape: {df.shape}")
print("\nColumns:", list(df.columns))
print("\nCategory distribution:")
print(df["category"].value_counts())

print("\nSentiment distribution:")
print(df["sentiment"].value_counts())

# Quick null check
print("\nMissing values per column:\n", df.isnull().sum())

# Preview few rows
df.sample(3, random_state=42)


Loaded dataset from: /home/niranjanrao07/cod-multiagent-ecommerce/notebooks/data/processed/amazon_sentiment_subsets/amazon_sentiment_cleaned.parquet
Shape: (133730, 10)

Columns: ['rating', 'text', 'title', 'asin', 'user_id', 'timestamp', 'verified_purchase', 'helpful_vote', 'category', 'sentiment']

Category distribution:
category
All_Beauty     49812
Electronics    49043
Books          34875
Name: count, dtype: int64

Sentiment distribution:
sentiment
positive    105003
negative     17203
neutral      11524
Name: count, dtype: int64

Missing values per column:
 rating               0
text                 0
title                0
asin                 0
user_id              0
timestamp            0
verified_purchase    0
helpful_vote         0
category             0
sentiment            0
dtype: int64


,rating,text,title,asin,user_id,timestamp,verified_purchase,helpful_vote,category,sentiment
18158,3.0,"just to small, likes it.",order not confusing.,B08H4PNS26,AH6R4FLHXNR4KWC5SWK5ILMYOMFQ,1609124593518,True,0,All_Beauty,neutral
12601,5.0,the battery last a good amount of time and the...,professional use.,B00692OA2M,AELD3RRLOKVKKW6BWYGIE4EN6Y4Q,1454040843000,True,1,All_Beauty,positive
14106,5.0,totally impressed. i went around and misted ev...,super cool!,B07C533XCW,AHM6MNZ4K7XBAR6B5SNNF2AIY7SA,1556625322412,True,0,All_Beauty,positive


- The dataset loaded cleanly (133,730 rows, 10 columns).
- Sentiment labels were regenerated properly.
- No missing values, it’s model-ready structurally.
- Category and sentiment distributions look balanced enough for prototyping.

In [4]:

#Text Normalization & Cleaning

import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Download resources (only first run)
nltk.download('stopwords')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    # Lowercase
    text = text.lower()
    # Remove punctuation, numbers, and special characters
    text = re.sub(r'[^a-z\s]', '', text)
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    # Remove stopwords and lemmatize
    tokens = [lemmatizer.lemmatize(w) for w in text.split() if w not in stop_words]
    return ' '.join(tokens)

# Apply cleaning to review text
df["clean_text"] = df["text"].apply(clean_text)

print("Example before & after cleaning:\n")
sample_idx = df.sample(1, random_state=42).index[0]
print("Original:", df.loc[sample_idx, "text"])
print("Cleaned:", df.loc[sample_idx, "clean_text"])

# Quick length check
df["clean_length"] = df["clean_text"].apply(lambda x: len(x.split()))
print("\nAverage tokens per review:", round(df["clean_length"].mean(), 2))


[nltk_data] Downloading package stopwords to
[nltk_data]     /home/niranjanrao07/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /home/niranjanrao07/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Example before & after cleaning:

Original: just to small, likes it.
Cleaned: small like

Average tokens per review: 41.08
